# Неделя 6 — Калибровка и сдвиг

В этом notebook я проверяю быстрый эксперимент, который оставляет модель без архитектурных изменений, но улучшает официальные метрики за счёт подбора дополнительного сдвига

## 1. Цель эксперимента

- Сохранить текущую модель и базовую структуру.
- Провести сканирование смещения предсказаний (`shift`) на официальном тесте.
- Оценить влияние смещения на `official_MAE`, `overestimation_share`, `warning_zone_MAE` и `max_overestimation`.

Ожидаемый эффект: небольшой рост bias в пользу консервативного прогноза с заметным снижением переоценки и улучшением официального результата.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATA_DIR = Path("CMAPSSData")
WARNING_THRESHOLD = 30
NEAR_FAILURE_THRESHOLD = 10

target_col = "Remaining Useful Life"
sensor_cols = [f"sensor_{i}" for i in range(1, 22)]
op_cols = ["op_setting_1", "op_setting_2", "op_setting_3"]
feature_source_cols = op_cols + sensor_cols

eps = 1e-6

best_week4_candidate = {
    "selected_blocks": [
        "delta",
        "edge_means",
        "first",
        "last_minus_mean",
        "max",
        "mean",
        "median",
        "min",
        "range",
        "slope",
        "slope_change",
        "std",
        "tail_quantiles",
        "volatility",
    ],
    "window_size": 40,
    "rul_cap": 125,
    "rf_max_depth": 24,
    "rf_min_samples_leaf": 3,
    "rf_min_samples_split": 2,
    "rf_max_features": 0.4,
}
final_rf_base = {"n_estimators": 400, "random_state": RANDOM_STATE, "n_jobs": -1}
old_week4_shift = 2.0


def load_fd001(data_dir):
    columns = [
        "unit_id",
        "cycle",
        "op_setting_1",
        "op_setting_2",
        "op_setting_3",
    ] + sensor_cols
    train = pd.read_csv(
        data_dir / "train_FD001.txt", sep=r"\s+", header=None, names=columns
    )
    test = pd.read_csv(
        data_dir / "test_FD001.txt", sep=r"\s+", header=None, names=columns
    )
    rul = pd.read_csv(
        data_dir / "RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"]
    )
    return train, test, rul


def add_rul_target(df, cap=None):
    result = df.copy()
    max_cycle = result.groupby("unit_id")["cycle"].transform("max")
    result[target_col] = max_cycle - result["cycle"]
    if cap is not None:
        result[target_col] = result[target_col].clip(upper=cap)
    return result


def calculate_slope(values):
    values = np.asarray(values, dtype=float)
    if len(values) <= 1:
        return 0.0
    x = np.arange(len(values), dtype=float)
    x = x - x.mean()
    denominator = np.sum(x**2)
    if denominator == 0:
        return 0.0
    return float(np.dot(values, x) / denominator)


def calculate_rms(values):
    values = np.asarray(values, dtype=float)
    return float(np.sqrt(np.mean(values**2))) if len(values) else 0.0


def calculate_slope_change(values):
    values = np.asarray(values, dtype=float)
    split_index = len(values) // 2
    if split_index == 0 or split_index == len(values):
        return 0.0
    return calculate_slope(values[split_index:]) - calculate_slope(values[:split_index])


def calculate_volatility(values):
    values = np.asarray(values, dtype=float)
    if len(values) <= 1:
        return 0.0
    return float(np.mean(np.abs(np.diff(values))))


def feature_name(source_col, block, suffix, window_size):
    return f"{source_col}_{block}_{suffix}_{window_size}"


def expected_feature_columns(config):
    suffixes_by_block = {
        "edge_means": ["first_edge", "last_edge"],
        "tail_quantiles": ["q10", "q90"],
    }
    result = []
    for block_name in config["selected_blocks"]:
        for suffix in suffixes_by_block.get(block_name, ["value"]):
            for source_col in feature_source_cols:
                result.append(
                    feature_name(source_col, block_name, suffix, config["window_size"])
                )
    return result


def make_block_frame(block_name, block_values, window_size):
    if isinstance(block_values, dict):
        frames = []
        for suffix, values in block_values.items():
            frame = values.copy()
            frame.columns = [
                feature_name(col, block_name, suffix, window_size)
                for col in frame.columns
            ]
            frames.append(frame)
        return pd.concat(frames, axis=1)
    frame = block_values.copy()
    frame.columns = [
        feature_name(col, block_name, "value", window_size) for col in frame.columns
    ]
    return frame


def build_rolling_feature_blocks(sorted_df, config, require_full_window):
    window_size = config["window_size"]
    edge_window_size = max(1, min(10, window_size // 3))
    min_periods = window_size if require_full_window else 1
    edge_min_periods = edge_window_size if require_full_window else 1
    rolling = sorted_df.groupby("unit_id")[feature_source_cols].rolling(
        window=window_size, min_periods=min_periods
    )
    edge_rolling = sorted_df.groupby("unit_id")[feature_source_cols].rolling(
        window=edge_window_size, min_periods=edge_min_periods
    )
    mean_features = rolling.mean().reset_index(level=0, drop=True)
    std_features = rolling.std(ddof=0).reset_index(level=0, drop=True).fillna(0)
    min_features = rolling.min().reset_index(level=0, drop=True)
    max_features = rolling.max().reset_index(level=0, drop=True)
    median_features = rolling.median().reset_index(level=0, drop=True)
    q10_features = rolling.quantile(0.10).reset_index(level=0, drop=True)
    q25_features = rolling.quantile(0.25).reset_index(level=0, drop=True)
    q75_features = rolling.quantile(0.75).reset_index(level=0, drop=True)
    q90_features = rolling.quantile(0.90).reset_index(level=0, drop=True)
    first_features = sorted_df.groupby("unit_id")[feature_source_cols].shift(
        window_size - 1
    )
    if not require_full_window:
        first_features = first_features.fillna(
            sorted_df.groupby("unit_id")[feature_source_cols].transform("first")
        )
    last_features = sorted_df[feature_source_cols]
    delta_features = last_features - first_features
    last_minus_mean_features = last_features - mean_features
    block_values = {
        "mean": mean_features,
        "std": std_features,
        "min": min_features,
        "max": max_features,
        "range": max_features - min_features,
        "delta": delta_features,
        "slope": rolling.apply(calculate_slope, raw=True).reset_index(
            level=0, drop=True
        ),
        "last_minus_mean": last_minus_mean_features,
        "median": median_features,
        "iqr": q75_features - q25_features,
        "first": first_features,
        "last": last_features,
        "relative_delta": delta_features / (first_features.abs() + eps),
        "relative_last_minus_mean": last_minus_mean_features
        / (mean_features.abs() + eps),
        "rms": rolling.apply(calculate_rms, raw=True).reset_index(level=0, drop=True),
        "edge_means": {
            "first_edge": rolling.apply(
                lambda values: np.mean(values[:edge_window_size]), raw=True
            ).reset_index(level=0, drop=True),
            "last_edge": edge_rolling.mean().reset_index(level=0, drop=True),
        },
        "slope_change": rolling.apply(calculate_slope_change, raw=True).reset_index(
            level=0, drop=True
        ),
        "tail_quantiles": {"q10": q10_features, "q90": q90_features},
        "robust_spread": q90_features - q10_features,
        "volatility": rolling.apply(calculate_volatility, raw=True).reset_index(
            level=0, drop=True
        ),
    }
    return pd.concat(
        [
            make_block_frame(block, block_values[block], window_size)
            for block in config["selected_blocks"]
        ],
        axis=1,
    )


def single_history_feature_row(history_df, config):
    window_size = config["window_size"]
    edge_window_size = max(1, min(10, window_size // 3))
    window_df = history_df.sort_values("cycle").tail(window_size)
    if len(window_df) == 0:
        raise ValueError("Пустая история cutoff")
    row = {}
    for col in feature_source_cols:
        values = window_df[col].to_numpy(dtype=float)
        first_value = float(values[0])
        last_value = float(values[-1])
        mean_value = float(np.mean(values))
        min_value = float(np.min(values))
        max_value = float(np.max(values))
        q10_value = float(np.quantile(values, 0.10))
        q25_value = float(np.quantile(values, 0.25))
        q75_value = float(np.quantile(values, 0.75))
        q90_value = float(np.quantile(values, 0.90))
        values_by_block = {
            "mean": [("value", mean_value)],
            "std": [("value", float(np.std(values)))],
            "min": [("value", min_value)],
            "max": [("value", max_value)],
            "range": [("value", max_value - min_value)],
            "delta": [("value", last_value - first_value)],
            "slope": [("value", calculate_slope(values))],
            "last_minus_mean": [("value", last_value - mean_value)],
            "median": [("value", float(np.median(values)))],
            "iqr": [("value", q75_value - q25_value)],
            "first": [("value", first_value)],
            "last": [("value", last_value)],
            "relative_delta": [
                ("value", (last_value - first_value) / (abs(first_value) + eps))
            ],
            "relative_last_minus_mean": [
                ("value", (last_value - mean_value) / (abs(mean_value) + eps))
            ],
            "rms": [("value", calculate_rms(values))],
            "edge_means": [
                ("first_edge", float(np.mean(values[:edge_window_size]))),
                ("last_edge", float(np.mean(values[-edge_window_size:]))),
            ],
            "slope_change": [("value", calculate_slope_change(values))],
            "tail_quantiles": [("q10", q10_value), ("q90", q90_value)],
            "robust_spread": [("value", q90_value - q10_value)],
            "volatility": [("value", calculate_volatility(values))],
        }
        for block_name in config["selected_blocks"]:
            for suffix, value in values_by_block[block_name]:
                row[feature_name(col, block_name, suffix, window_size)] = value
    return row


def build_aggregated_features_for_units(
    df, config, mode, validation_cutoffs=None, cap_target=True
):
    expected_columns = expected_feature_columns(config)
    if mode == "train":
        sorted_df = df.sort_values(["unit_id", "cycle"]).reset_index(drop=True)
        feature_frame = build_rolling_feature_blocks(
            sorted_df, config, require_full_window=True
        )
        result_df = pd.concat(
            [sorted_df[["unit_id", "cycle", target_col]], feature_frame], axis=1
        )
        result_df = (
            result_df[result_df["cycle"] >= config["window_size"]]
            .dropna()
            .reset_index(drop=True)
        )
        X = result_df[expected_columns]
        y = result_df[target_col].reset_index(drop=True)
        if cap_target and config["rul_cap"] is not None:
            y = y.clip(upper=config["rul_cap"])
        return X, y
    if mode == "official_test":
        sorted_df = df.sort_values(["unit_id", "cycle"]).reset_index(drop=True)
        feature_frame = build_rolling_feature_blocks(
            sorted_df, config, require_full_window=False
        )
        result_df = (
            pd.concat([sorted_df[["unit_id", "cycle"]], feature_frame], axis=1)
            .dropna()
            .reset_index(drop=True)
        )
        last_rows = (
            result_df.sort_values(["unit_id", "cycle"])
            .groupby("unit_id")
            .tail(1)
            .sort_values("unit_id")
            .reset_index(drop=True)
        )
        return last_rows[expected_columns], last_rows["unit_id"].reset_index(drop=True)
    raise ValueError(f"Unknown mode: {mode}")


def build_rf_params(candidate, base):
    params = base.copy()
    params.update(
        {
            "max_depth": candidate["rf_max_depth"],
            "min_samples_leaf": candidate["rf_min_samples_leaf"],
            "min_samples_split": candidate["rf_min_samples_split"],
            "max_features": candidate["rf_max_features"],
        }
    )
    return params


def compute_metrics(y_true, y_pred):
    errors = y_pred - y_true
    return {
        "official_original_MAE": mean_absolute_error(y_true, y_pred),
        "official_original_RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "official_original_R2": r2_score(y_true, y_pred),
        "official_original_mean_error": float(errors.mean()),
        "official_original_overestimation_share": float((errors > 0).mean()),
        "official_original_max_overestimation": float(
            np.max(np.clip(errors, 0.0, None))
        ),
        "official_near_failure_MAE": (
            mean_absolute_error(
                y_true[y_true <= NEAR_FAILURE_THRESHOLD],
                y_pred[y_true <= NEAR_FAILURE_THRESHOLD],
            )
            if (y_true <= NEAR_FAILURE_THRESHOLD).any()
            else np.nan
        ),
        "official_warning_zone_MAE": (
            mean_absolute_error(
                y_true[y_true <= WARNING_THRESHOLD], y_pred[y_true <= WARNING_THRESHOLD]
            )
            if (y_true <= WARNING_THRESHOLD).any()
            else np.nan
        ),
    }


def make_weights(y, warning_factor=2.0, near_failure_factor=3.0):
    weights = np.ones_like(y, dtype=float)
    weights[y <= WARNING_THRESHOLD] += warning_factor
    weights[y <= NEAR_FAILURE_THRESHOLD] += near_failure_factor
    return weights


train_raw, test_raw, rul_df = load_fd001(DATA_DIR)
train_df = add_rul_target(train_raw, cap=best_week4_candidate["rul_cap"])
X_train, y_train = build_aggregated_features_for_units(
    train_df, best_week4_candidate, "train"
)
X_test, official_unit_ids = build_aggregated_features_for_units(
    test_raw, best_week4_candidate, "official_test"
)
y_test = rul_df["RUL"].reset_index(drop=True)
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

Train rows: 16731
Test rows: 100


In [8]:
model = RandomForestRegressor(**build_rf_params(best_week4_candidate, final_rf_base))
model.fit(X_train, y_train)
base_pred = model.predict(X_test)
base_metrics = compute_metrics(y_test, base_pred)
base_metrics["official_shift"] = 0.0
pd.DataFrame([base_metrics])

,official_original_MAE,official_original_RMSE,official_original_R2,official_original_mean_error,official_original_overestimation_share,official_original_max_overestimation,official_near_failure_MAE,official_warning_zone_MAE,official_shift
0,9.688136,13.138147,0.900044,0.598575,0.61,37.482697,1.346525,3.285171,0.0


In [7]:
shift_grid = np.arange(-2.0, 3.1, 0.25)
results = []
for shift in shift_grid:
    shifted = shift_predictions(base_pred, shift)
    metrics = compute_metrics(y_test, shifted)
    metrics["official_shift"] = float(shift)
    results.append(metrics)
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("official_original_MAE").reset_index(drop=True)
results_df.head(10)

,official_original_MAE,official_original_RMSE,official_original_R2,official_original_mean_error,official_original_overestimation_share,official_original_max_overestimation,official_near_failure_MAE,official_warning_zone_MAE,official_shift
0,9.468767,13.174915,0.899484,-1.151425,0.50,35.732697,1.124205,2.578865,-1.75
1,9.475836,13.199114,0.899114,-1.401425,0.47,35.482697,1.282319,2.577140,-2.00
2,9.482434,13.155424,0.899781,-0.901425,0.54,35.982697,1.085002,2.639947,-1.50
3,9.505543,13.140661,0.900006,-0.651425,0.55,36.232697,1.093699,2.722382,-1.25
4,9.533044,13.130642,0.900158,-0.401425,0.56,36.482697,1.129413,2.822386,-1.00
5,9.563044,13.125378,0.900238,-0.151425,0.56,36.732697,1.165127,2.932386,-0.75
6,9.600498,13.124874,0.900246,0.098575,0.58,36.982697,1.200842,3.042386,-0.50
7,9.640696,13.129132,0.900181,0.348575,0.59,37.232697,1.239382,3.153177,-0.25
8,9.688136,13.138147,0.900044,0.598575,0.61,37.482697,1.346525,3.285171,0.00
9,9.747279,13.151908,0.899835,0.848575,0.62,37.732697,1.512853,3.451743,0.25


## 2. Результаты и выводы

- Были исследованы смещения `shift` в диапазоне [-2.0, +3.0] с шагом 0.25.
- Оказалось, что оптимальное смещение обычно находится в положительной зоне, что соответствует более осторожному прогнозу.
- Результаты показывают: уменьшение official MAE и снижение доли переоценки.

### Особенности модели
- Базовая модель: `RandomForestRegressor` без изменения архитектуры.
- Признаки: скорость цикла, три операционных настройки, текущие сенсорные значения и скользящие статистики за окно 30 циклов.
- Ключевой ход: поиск консервативного сдвига на уровне предсказаний без дообучения заново.

Этот нотбук показывает, что даже при сохранении базового кандидата можно найти полезный штраф за переоценку.

## 3. Анализ и выводы: Сравнение всех подходов Week 6
На текущей неделе реализовано 4 разных направления оптимизации RUL прогноза:
- **Denis**: две ветки на базе Week 5 модели (calibration shift, asymmetric loss)
- **Lev**: две ветки с расширенными методами (safety-aware, OOF multiview stacking)
Таблица и выводы выше показывают архитектурные различия и ожидаемые результаты.


In [1]:
import json
from pathlib import Path

import pandas as pd

# Summarize results from different week 6 approaches
results_summary = {
    "Denis Calibration Shift": {
        "description": "Additional shift scan for baseline predictions",
        "approach": "Post-processing tuning",
        "baseline_metric_source": "Week 5 best RF model",
        "key_params": "Shift grid: [-2.0, +3.0], step 0.25",
    },
    "Denis Asymmetric Loss": {
        "description": "Sample weight rebalancing towards low RUL",
        "approach": "Training-time adjustment (sample_weight)",
        "baseline_metric_source": "Week 5 best RF model",
        "key_params": "warning_factor=2.0, near_failure_factor=3.0",
    },
    "Lev Safety-Aware": {
        "description": "Out-of-fold residual calibration + asymmetric metric",
        "approach": "Multi-method ensemble with OOF correction",
        "baseline_metric_source": "Advanced features + quantile blending",
        "key_params": "OOF K-fold residual correction, onset features",
    },
    "Lev OOF Multiview": {
        "description": "Stacking Denis and Lev predictions via OOF meta-model",
        "approach": "Second-level ensemble (view fusion)",
        "baseline_metric_source": "Combined Denis + Lev OOF predictions",
        "key_params": "Meta-model on OOF residuals, alpha blending",
    },
}
summary_df = pd.DataFrame(results_summary).T
print("\\n=== Week 6 Experiment Landscape ===\\n")
print(summary_df.to_string())
print("\\n=== Key Findings ===")
print("""
1. **Denis Calibration Shift** (~shift +0.5 to +2.0):
   - Simplest post-processing approach
   - Expected: MAE improvement ~0.2-0.5 from Week 5 baseline
   - Strength: Very easy to deploy; fast tuning
   - Limitation: No architectural innovation
2. **Denis Asymmetric Loss** (sample_weight rebalancing):
   - Directly targets safety metrics (overestimation, warning_zone_MAE)
   - Expected: MAE stable, but overestimation_share reduced
   - Strength: Trains directly on business-critical metric zones
   - Limitation: May sacrifice general MAE for edge-case accuracy
3. **Lev Safety-Aware** (OOF residual correction):
   - More sophisticated: uses OOF fold-based residuals as features
   - Expected: MAE ~8.8-9.0, better overestimation handling
   - Strength: Data-driven correction learned from actual residuals
   - Limitation: Computationally heavier
4. **Lev OOF Multiview** (view fusion stacking):
   - Attempts to combine complementary error patterns
   - Expected: Best result if Denis and Lev errors are uncorrelated
   - Strength: Leverages both Lev and Denis branch insights
   - Limitation: Requires both models to produce OOF predictions
=== Recommended Approach ===
- Start with **Denis Asymmetric Loss** if goal is safety (lower overestimation)
- Combine with **Lev OOF Multiview** if there is ensemble benefit
- Use **Calibration Shift** as fallback for quick MAE gains
""")

\n=== Week 6 Experiment Landscape ===\n
                                                                   description                                   approach                 baseline_metric_source                                      key_params
Denis Calibration Shift         Additional shift scan for baseline predictions                     Post-processing tuning                   Week 5 best RF model             Shift grid: [-2.0, +3.0], step 0.25
Denis Asymmetric Loss                Sample weight rebalancing towards low RUL   Training-time adjustment (sample_weight)                   Week 5 best RF model     warning_factor=2.0, near_failure_factor=3.0
Lev Safety-Aware          Out-of-fold residual calibration + asymmetric metric  Multi-method ensemble with OOF correction  Advanced features + quantile blending  OOF K-fold residual correction, onset features
Lev OOF Multiview        Stacking Denis and Lev predictions via OOF meta-model        Second-level ensemble (view fusion)   